**Engineering foundation branch.** Completed motion helpers and fixed experiment profiles are imported from this repository. The course discussion prompts below remain reference material. Training, scored simulation, and hardware approval are separate statuses.

# CS 123 — Sit, Stay, Shake 🐾

Lab 6, Step 2. You get two motions: Pupper **sits down and stands up**, and
Pupper **shakes its left paw** while sitting. You cut and join them into
five **clips**, then train one policy (BeyondMimic-style) that can play all
of them:

```
sit_down → sit_idle (loops: the dog stays) → shake_left / shake_right
           (each returns to sit_idle) → stand_up
```

You don't design any new poses. You write four small pieces of code: cut,
hold, mirror, and the clip list. Then you watch the plain motion fall over in
simulation, train a policy, and watch the policy stay up.

Prefer working on your laptop? The same TODOs are in
`tricks/sit_stay_shake.py` in pupper-mjlab.

*Runtime → Change runtime type → a GPU runtime (G4 recommended), then run cells top to bottom.*

## 1. Setup

In [ ]:
# Engineering branch: clone once, record its exact revision, and use repo code.
REPO_URL = "https://github.com/ShrishPremkrishna/mts_123.git"
BRANCH = "pupper-foundation"
PINNED_COMMIT = ""  # optionally paste the reviewed 40-character commit SHA

import argparse
import os
import subprocess
import sys
from pathlib import Path

checkout = Path("/content/cs123-foundation")
if not checkout.exists():
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "--filter=blob:none", "--sparse", REPO_URL, str(checkout)], check=True, env=env)
    subprocess.run(["git", "-C", str(checkout), "sparse-checkout", "set", "pupper-mjlab"], check=True, env=env)
origin = subprocess.check_output(["git", "-C", str(checkout), "remote", "get-url", "origin"], text=True).strip()
assert origin == REPO_URL, f"Existing checkout belongs to {origin}; use a fresh runtime."
if PINNED_COMMIT:
    subprocess.run(["git", "-C", str(checkout), "fetch", "origin", PINNED_COMMIT], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", PINNED_COMMIT], check=True)
else:
    branch = subprocess.check_output(["git", "-C", str(checkout), "branch", "--show-current"], text=True).strip()
    assert branch == BRANCH, f"Existing checkout is {branch}; pin an exact commit or use a fresh runtime."
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print("Training source revision:", revision)
os.chdir(checkout / "pupper-mjlab")
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(checkout / "pupper-mjlab" / "src"))
assert not any(k == "mjlab" or k.startswith("mjlab.") for k in sys.modules), "Restart runtime before changing source revisions."

# Set PUPPER_ARTIFACT_ROOT to an already-mounted persistent directory when available.
# The default /content directory is temporary; W&B also receives checkpoints/policies.
ARTIFACT_ROOT = Path(os.environ.get("PUPPER_ARTIFACT_ROOT", "/content/foundation-artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Artifacts:", ARTIFACT_ROOT, "(temporary unless configured to mounted storage)")

def run_foundation(profile, motion=None, iterations=None, num_envs=None):
    from mjlab.scripts.pupper_foundation import train
    return train(argparse.Namespace(profile=profile, output=ARTIFACT_ROOT, motion=motion,
        iterations=iterations, num_envs=num_envs, seed=42, logger="wandb", project="mjlab",
        resume_run=None, resume_checkpoint=None))


## 2. Log into Weights & Biases

W&B carries your motion from this notebook to training, and your trained policy
from training to the robot. **Use the same account your Pupper is logged into**
(the one you deploy walking policies with).

In [ ]:
import wandb
wandb.login()  # interactive/Colab Secrets authentication; never paste keys in source


## 3. What you are given

A **Motion** is three arrays, one row per frame, 50 frames per second
(frame `i` is at time `i / 50` s):

| array | shape | what |
| --- | --- | --- |
| `root_pos` | (T, 3) | where the body is, in metres (x forward, y left, z up) |
| `root_quat` | (T, 4) | how the body is turned (quaternion w, x, y, z) |
| `joint_pos` | (T, 12) | the 12 joint angles, in `md.JOINT_NAMES` order |

Make a new one with `md.Motion(fps=..., root_pos=..., root_quat=..., joint_pos=...)`.

Your two motions:
- `whole`: sit down, **sitting from 3.7 s to 5.7 s**, stand up.
- `left`: while sitting, shake the left paw, then sit still again.

In [ ]:
from pathlib import Path

import numpy as np

from mjlab.tasks.tracking.config.pupper import motion_design as md
from mjlab.tasks.tracking.config.pupper.tricks import sit_stay_shake as t

TRICK_NAME = "sit_stay_shake"  # the name on W&B and on the robot
OUT = Path("/content/tricks")
OUT.mkdir(exist_ok=True)

whole = t.whole_sit()
left = t.shake_left()
print(f"whole_sit: {whole.duration:.1f} s, {len(whole.joint_pos)} frames")
print(f"shake_left: {left.duration:.1f} s, {len(left.joint_pos)} frames")
print("joint order:", md.JOINT_NAMES)

Want to see them? This plays the motion with no physics (opens a public
link; stop the cell when done). Change `whole` to `left` to see the paw.

In [ ]:
MOTION_PREVIEW = whole.save_npz(OUT / "preview.npz")
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_PREVIEW} --no-physics True --viewer viser --share True

## 4. Stitch the clip library ✏️

### TODO 1: `cut`
Return the piece of a motion from `start_s` to `end_s`. **Keep both end
frames**: then a clip that ends at 3.7 s and a clip that starts at 3.7 s
share a frame, and they join with no jump.

In [ ]:
from mjlab.tasks.tracking.config.pupper.motion_ops import cut


### TODO 2: `hold`
The **stay**: stand still in the pose `motion` has at time `at_s`, for
`seconds`. Repeat that one frame with `np.repeat`, `seconds * 50 + 1` times.

In [ ]:
from mjlab.tasks.tracking.config.pupper.motion_ops import hold


### TODO 3: `mirror`
Swap left and right: `mirror(left)` shakes the **right** paw. Three changes:
- body position: flip the sign of y;
- body quaternion: `(w, x, y, z) → (w, −x, y, −z)`;
- joints: each left leg gets the right leg's angles, and the other way round.

For the joints, look at how Pupper's left and right motors are mounted (your
lab 2 kinematics helps). Copy the angles as they are, or flip some signs?
Which ones? The checks below will tell you.

In [ ]:
from mjlab.tasks.tracking.config.pupper.motion_ops import mirror


Check your code (you want all `ok`, no `FAIL`):

In [ ]:
assert t.check_helpers(), "Motion helper checks failed"


### TODO 4: the clip table and the training order

Make five clips with `md.Clip(name, motion, loop=..., next=...)`:
`sit_down`, `sit_idle`, `shake_left`, `shake_right`, `stand_up`.
- `loop=True` means the clip repeats until something else is asked for. Which one should loop?
- `next` is the clip that plays when this one ends. `stand_up` has `next=None`.

Then the **training order**: the clips are joined in this order for training.
The policy only learns the clip-to-clip changes it sees here, so include
every change the robot can make (like `sit_idle` → `shake_right`). Names can
repeat.

In [ ]:
from mjlab.tasks.tracking.config.pupper.tricks.sit_stay_shake import clips, ORDER

library = md.concat_clips(clips(), order=ORDER)
for key, value in library.report.items():
  print(f"{key:36} {value}")
for c in library.clips:
  print(f"  clip {c['name']:12} frames {c['start']:4}-{c['end']:4} "
        f"({(c['end'] - c['start']) / 50:.1f} s) loop={c['loop']} next={c['next']}")
MOTION_FILE = library.save_npz(OUT / f"{TRICK_NAME}.npz")
print("saved", MOTION_FILE)

## 4. Replay it with physics

Plays the joint trajectory on the simulated robot with the robot's
trick-player gains (no policy) and prints the tracking error and a verdict:

- **HOLDS UP**: plain replay works. Train anyway: on the robot every trick runs as a policy, and a policy also catches slips and pushes.
- **WOBBLES**: it stays up but drifts. A policy will be tighter.
- **FALLS**: plain replay won't work. Make the motion gentler, or train it.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE}

Watch where it goes wrong (public link; stop the cell when done). The see-through
ghost is the motion it should follow.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --viewer viser --share True

**Deliverable:** the verdict and the worst errors above. In which clip does it
fall? Why can't a fixed recording catch itself?

## 5. Upload the motion to W&B

One upload, then everything refers to it by name
(`<your-entity>/mjlab/sit_stay_shake`). Re-uploading after a change makes a
new version; training uses the latest.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.upload_motion {MOTION_FILE} --name {TRICK_NAME}

MOTION_ARTIFACT = f"{wandb.Api().default_entity}/mjlab/{TRICK_NAME}"
print("motion artifact:", MOTION_ARTIFACT)

## 6. Train 🚀

3,000 iterations. Open the printed W&B link to watch.
The numbers that matter are `Metrics/motion/error_*` (lower is better) and
`Episode_Reward/motion_*` (close to 1 is good).

The trick policy is uploaded to the run as `policy.json` at every checkpoint
and when you press stop, and the log prints the exact command for the robot.

In [ ]:
RUN_DIR = run_foundation("sit_stay_shake")
MOTION_FILE = RUN_DIR / "sit_stay_shake.npz"


## 7. Score it

The same physics check as section 4, now with your policy in charge. Compare
the verdict and the worst errors with plain replay: that is what training
bought you.

In [ ]:
CHECKPOINT = max(RUN_DIR.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("checkpoint:", CHECKPOINT)
subprocess.run([sys.executable, "-m", "mjlab.scripts.pupper_foundation", "score",
    "--motion", str(MOTION_FILE), "--checkpoint", str(CHECKPOINT),
    "--output", str(RUN_DIR / "evaluation"), "--suite", "--stress-profile", "sit_stay_shake"], check=True)


Watch the policy (public link; stop the cell when done):

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --checkpoint {CHECKPOINT} --viewer viser --share True --loops 3

## 8. On the robot 🤖

**TA gate first:** show a TA the score from section 7 and the viewer.

Then, in your gemini_pupper repo on the Pupper (logged into the same W&B
account), fetch the policy with the command the training log printed:

```bash
python3 robot/get_trick.py <entity>/mjlab/<run-id> --name sit_stay_shake
```

Fill in `tricks/sit_stay_shake/trick.yaml`:
- `description`: what Gemini reads when deciding whether this is the trick someone asked for;
- `actions`: the clips Gemini may request while it sits, each with a description Gemini will read;
- `exit_clip`: the clip that brings Pupper back to standing;
- `end_pose: sitting`, and `max_body_angle_deg: 60` (the sit leans back about 35 degrees).

Restart Gemini Pupper (`robot/stop.sh`, then `robot/start.sh`) and ask it to sit.

Safety: robot on the ground, clear space around it, a hand near the e-stop.
When you're done here: **Runtime → Disconnect**.